# Preconditioned MALA (pMALA) Tutorial

This tutorial demonstrates `ls_bayesian`'s function-space preconditioned MALA sampler,
`PMALAAlgorithm` (`ls_bayesian.mcmc.algorithms.pmala`): the state-independent special case of
$\infty$-mMALA (Beskos, Girolami, Lan, Farrell, Stuart, 2017) that preconditions Langevin dynamics
with a fixed Gaussian $\overline K$, distinct from the target's own reference covariance $C$. As in
the `pcn` tutorial, we use a linear-Gaussian toy inverse problem whose posterior is known in closed
form, so every sampler run can be checked against an exact reference -- this time also
demonstrating why a well-chosen preconditioner matters for a target whose posterior geometry
differs substantially from its prior.

## Mathematical Formulation

`PMALAAlgorithm` keeps the target's potential $\Phi$ relative to the true reference measure $\mu_0
= \mathcal N(0, C)$ throughout -- unlike pCN's generalization, which instead reproposes from an
alternative Gaussian. Beskos et al. (2017) show that a location-specific Gaussian $K(u)$ can
precondition the Langevin dynamics; fixing $K(u) \equiv \overline K$ (e.g. a Laplace approximation
computed once at the MAP, rather than re-linearized at every state) keeps the acceptance
probability tractable while still adapting the proposal to a target geometry that may differ
substantially from $\mu_0$'s.

Given the current state $u$, define the score $s(u) = \nabla\Phi(u) + C^{-1}u$ (the gradient of the
full target's log-density, combining $\Phi$'s gradient with the reference's own precision term).
The proposal is

$$
v = u - \frac{2\delta}{2+\delta}\overline K s(u) + \frac{\sqrt{8\delta}}{2+\delta}\, w, \qquad w
\sim \mathcal N(0, \overline K),
$$

With $\rho = \frac{2-\delta}{2+\delta}$, $g(u) = u - \overline K s(u)$, and innovation $w(u,v) = (v- \rho u) / \sqrt{1-\rho^2}$, the acceptance probability is $\alpha(u,v) = 1 \wedge
\exp(\varrho(u,v) - \varrho(v,u))$, where

$$
\varrho(u,v) = \Phi(u) + \frac{1}{2}\Big\langle \sqrt{\delta/2}\, g(u) - w(u,v),\
    \overline K^{-1}\big(\sqrt{\delta/2}\, g(u) - w(u,v)\big) \Big\rangle - \frac{1}{2}
    \big\langle w(u,v), C^{-1} w(u,v) \big\rangle.
$$

With $\overline K = C$ (i.e. `approximation` equal to `reference`), this reduces exactly to plain
MALA (`MALAAlgorithm`); this tutorial first runs that plain-MALA-equivalent baseline, then an
informed $\overline K$ to show how a good preconditioner sustains high acceptance at much larger
step widths.

References:

- Beskos, Girolami, Lan, Farrell, Stuart (2017). *Geometric MCMC for Infinite-Dimensional Inverse
  Problems.* Journal of Computational Physics 335, 327-351.

## Imports and Configuration

We import NumPy for the linear algebra, Matplotlib (plus `matplotlib.patches.Ellipse` for covariance ellipses) for the diagnostic plots, `scipy.stats.norm` for the analytic 1D marginal density, `typing.override` for the measure/target implementations, and the `algorithms.pmala`, `measures`, `sampler`, and `output` modules of `ls_bayesian`'s `mcmc` subpackage. All randomness is seeded for reproducibility.

In [ ]:
from typing import override

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.stats import norm

from ls_bayesian.mcmc import output as mcmc_output
from ls_bayesian.mcmc.algorithms.pmala import PMALAAlgorithm
from ls_bayesian.mcmc.measures import DifferentiableTargetMeasure, GaussianMeasure
from ls_bayesian.mcmc.model import MCMCModel
from ls_bayesian.mcmc.sampler import Sampler, SamplerSettings
from ls_bayesian.mcmc.storage import NumpyStorage

rng = np.random.default_rng(0)
STATE_DIM = 4

## A Linear-Gaussian Test Problem

We reuse the same test problem as the `pcn` tutorial: a quadratic potential
$\Phi(u) = \frac{1}{2}(u-a)^T H (u-a)$ combined with a centered Gaussian reference $\mu_0 =
\mathcal N(0, C)$, giving a Gaussian target with precision $H + C^{-1}$ and mean
$(H+C^{-1})^{-1} H a$. Here $H$ is deliberately built independently of $C$, so the posterior's
geometry differs substantially from the prior's -- exactly the situation in which a
preconditioner tuned to the posterior helps most.

In [ ]:
def random_spd_matrix(rng: np.random.Generator, dim: int) -> np.ndarray:
    """Return a random symmetric positive-definite matrix of shape (dim, dim)."""
    factor = rng.random((dim, dim))
    return factor @ factor.T + dim * np.eye(dim)


hessian = random_spd_matrix(rng, STATE_DIM)
prior_covariance = random_spd_matrix(rng, STATE_DIM)
minimizer = rng.standard_normal(STATE_DIM)

prior_precision = np.linalg.inv(prior_covariance)
posterior_precision = hessian + prior_precision
posterior_covariance = np.linalg.inv(posterior_precision)
posterior_mean = posterior_covariance @ (hessian @ minimizer)
posterior_standard_deviation = np.sqrt(np.diag(posterior_covariance))

print("Analytic posterior mean:", posterior_mean)
print("Analytic posterior marginal std :", posterior_standard_deviation)

## Implementing the Target and the Gaussian Measures

`PMALAAlgorithm` takes one `MCMCModel`: `target` (a `DifferentiableTargetMeasure` for $\Phi$ and
$\nabla\Phi$), `reference` (a `GaussianMeasure` for $\mu_0$'s precision $C^{-1}$), and
`approximation` (a `GaussianMeasure` for $\overline K$, used only for its covariance/precision
actions). `DenseGaussianMeasure` below implements `GaussianMeasure`, mirroring
`tests/mcmc/helpers.py`, so the same class can build $\mu_0$ and any candidate $\overline K$.

In [ ]:
class QuadraticTargetMeasure(DifferentiableTargetMeasure):
    """Quadratic potential Phi(u) = 1/2 (u-a)^T H (u-a), exact gradient H(u-a)."""

    def __init__(self, matrix: np.ndarray, minimizer: np.ndarray) -> None:
        self.matrix = matrix
        self.minimizer = minimizer

    @override
    def evaluate_potential(self, state: np.ndarray) -> float:
        difference = state - self.minimizer
        return float(0.5 * difference @ self.matrix @ difference)

    @override
    def evaluate_gradient(self, state: np.ndarray) -> np.ndarray:
        return self.matrix @ (state - self.minimizer)


class DenseGaussianMeasure(GaussianMeasure):
    """Centered Gaussian measure N(0, covariance), usable as mu_0 (`reference`) or as a
    preconditioner `Kbar` (`approximation`, `mean` unused in that role)."""

    def __init__(self, covariance_matrix: np.ndarray) -> None:
        self.covariance_matrix = covariance_matrix
        self.precision_matrix = np.linalg.inv(covariance_matrix)
        self.covariance_factor = np.linalg.cholesky(covariance_matrix)

    @property
    @override
    def mean(self) -> np.ndarray:
        return np.zeros(self.covariance_matrix.shape[0])

    @property
    @override
    def random_vector_size(self) -> int:
        return self.covariance_factor.shape[1]

    @override
    def apply_covariance_factorization(self, random_vector: np.ndarray) -> np.ndarray:
        return self.covariance_factor @ random_vector

    @override
    def apply_covariance_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.covariance_matrix @ vector

    @override
    def apply_precision_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ vector


target = QuadraticTargetMeasure(hessian, minimizer)
reference_measure = DenseGaussianMeasure(prior_covariance)

## Plain MALA Baseline: Preconditioning With the Prior Itself

Passing `reference_measure` as both `reference` and `approximation` sets $\overline K = C$,
recovering plain MALA. Because $H$ and $C^{-1}$ are unrelated random matrices here, the
posterior's geometry is poorly matched by $C$: a step width large enough to explore efficiently in
the prior's own scale is far too large once weighted by $H$, so the acceptance rate collapses
quickly as $\delta$ grows.

In [ ]:
BURN_IN = 2000
NUM_SAMPLES = 20_000


def run_pmala_chain(
    preconditioner: GaussianMeasure, step_width: float, seed: int
) -> tuple[np.ndarray, float]:
    """Run PMALAAlgorithm for BURN_IN + NUM_SAMPLES steps and return the post-burn-in samples and
    the final running-mean acceptance rate."""
    model = MCMCModel(target=target, reference=reference_measure, approximation=preconditioner)
    algorithm = PMALAAlgorithm(model, step_width)
    acceptance_output = mcmc_output.build(
        mcmc_output.AcceptanceQoI(), mcmc_output.RunningMeanStatistic()
    )
    storage = NumpyStorage()
    sampler = Sampler(algorithm, storage=storage, outputs=[acceptance_output])
    settings = SamplerSettings(
        num_samples=BURN_IN + NUM_SAMPLES, log_interval=BURN_IN + NUM_SAMPLES
    )
    sampler.run(posterior_mean.copy(), settings, seed=seed)
    return storage.values[BURN_IN:], acceptance_output.value


plain_samples, plain_acceptance_rate = run_pmala_chain(
    reference_measure, step_width=0.02, seed=1
)
print(f"Plain-MALA-equivalent acceptance rate (step_width=0.02): {plain_acceptance_rate:.3f}")

## Verifying the Recovered Posterior Moments

The empirical mean and standard deviation of the post-burn-in samples should match the analytic
posterior moments computed above, up to Monte Carlo error.

In [ ]:
def report_moment_recovery(samples: np.ndarray, label: str) -> None:
    """Print the sample mean/std next to the analytic reference, normalized by the analytic
    standard deviation so the comparison is scale-free across coordinates."""
    sample_mean = samples.mean(axis=0)
    sample_std = samples.std(axis=0, ddof=1)
    normalized_mean_error = (sample_mean - posterior_mean) / posterior_standard_deviation
    print(f"{label}:")
    print("  normalized mean error:", normalized_mean_error)
    print("  sample std / analytic std:", sample_std / posterior_standard_deviation)
    assert np.all(np.abs(normalized_mean_error) < 0.5)
    assert np.all(np.abs(sample_std / posterior_standard_deviation - 1.0) < 0.3)


report_moment_recovery(plain_samples, "Plain-MALA-equivalent pMALA")

## Trace Plot (Plain-MALA-equivalent pMALA)

Plotting each state component against sample index is the standard first check for a chain's mixing behavior: a well-mixing chain looks like stationary noise around the analytic mean (dashed line), with no visible drift or long excursions.

In [ ]:
fig, axes = plt.subplots(STATE_DIM, 1, figsize=(8, 2 * STATE_DIM), sharex=True)
for component, ax in enumerate(axes):
    ax.plot(plain_samples[:, component], linewidth=0.5)
    ax.axhline(posterior_mean[component], color="black", linestyle="--", linewidth=1)
    ax.set_ylabel(f"$u_{component}$")
axes[-1].set_xlabel("post-burn-in sample index")
fig.suptitle("Plain-MALA-equivalent pMALA: trace plot")
fig.tight_layout()

## 1D Marginals vs. Analytical (Plain-MALA-equivalent pMALA)

In [ ]:
fig, axes = plt.subplots(1, STATE_DIM, figsize=(4 * STATE_DIM, 3))
for component, ax in enumerate(axes):
    ax.hist(plain_samples[:, component], bins=50, density=True, alpha=0.6, label="samples")
    grid = np.linspace(*ax.get_xlim(), 200)
    analytic_density = norm.pdf(
        grid, posterior_mean[component], posterior_standard_deviation[component]
    )
    ax.plot(grid, analytic_density, color="black", label="analytic")
    ax.set_xlabel(f"$u_{component}$")
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("Plain-MALA-equivalent pMALA: 1D marginals vs. analytic Gaussian")
fig.tight_layout()

## Pairwise 2D Marginals vs. Analytical (Plain-MALA-equivalent pMALA)

In [ ]:
def add_covariance_ellipses(ax, mean_pair, covariance_pair, n_std_values=(1, 2)) -> None:
    """Draw n_std_values-sigma covariance ellipses of a 2D Gaussian with the given mean/covariance."""
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_pair)
    angle = np.degrees(np.arctan2(eigenvectors[1, -1], eigenvectors[0, -1]))
    for n_std in n_std_values:
        width, height = 2 * n_std * np.sqrt(eigenvalues[::-1])
        ax.add_patch(
            Ellipse(mean_pair, width, height, angle=angle, fill=False, edgecolor="black", lw=1)
        )


fig, axes = plt.subplots(STATE_DIM, STATE_DIM, figsize=(3 * STATE_DIM, 3 * STATE_DIM))
for row in range(STATE_DIM):
    for col in range(STATE_DIM):
        ax = axes[row, col]
        if row <= col:
            ax.axis("off")
            continue
        ax.scatter(plain_samples[:, col], plain_samples[:, row], s=2, alpha=0.15)
        add_covariance_ellipses(
            ax,
            posterior_mean[[col, row]],
            posterior_covariance[np.ix_([col, row], [col, row])],
        )
        if row == STATE_DIM - 1:
            ax.set_xlabel(f"$u_{col}$")
        if col == 0:
            ax.set_ylabel(f"$u_{row}$")
fig.suptitle("Plain-MALA-equivalent pMALA: pairwise 2D marginals vs. analytic covariance ellipses")
fig.tight_layout()

## Preconditioned pMALA: an Informed $\overline K$

Now we precondition with the posterior's own covariance instead (in practice, e.g., a Laplace
approximation computed once at the MAP estimate). This lets the proposal's drift and noise adapt
to the posterior's actual geometry, so a much larger step width remains stable and well-accepted.

In [ ]:
informed_preconditioner = DenseGaussianMeasure(posterior_covariance)

informed_samples, informed_acceptance_rate = run_pmala_chain(
    informed_preconditioner, step_width=1.0, seed=1
)
print(f"Preconditioned pMALA acceptance rate (step_width=1.0): {informed_acceptance_rate:.3f}")
report_moment_recovery(informed_samples, "Preconditioned pMALA")

## Trace Plot (Preconditioned pMALA)

Plotting each state component against sample index is the standard first check for a chain's mixing behavior: a well-mixing chain looks like stationary noise around the analytic mean (dashed line), with no visible drift or long excursions.

In [ ]:
fig, axes = plt.subplots(STATE_DIM, 1, figsize=(8, 2 * STATE_DIM), sharex=True)
for component, ax in enumerate(axes):
    ax.plot(informed_samples[:, component], linewidth=0.5)
    ax.axhline(posterior_mean[component], color="black", linestyle="--", linewidth=1)
    ax.set_ylabel(f"$u_{component}$")
axes[-1].set_xlabel("post-burn-in sample index")
fig.suptitle("Preconditioned pMALA: trace plot")
fig.tight_layout()

## 1D Marginals vs. Analytical (Preconditioned pMALA)

In [ ]:
fig, axes = plt.subplots(1, STATE_DIM, figsize=(4 * STATE_DIM, 3))
for component, ax in enumerate(axes):
    ax.hist(informed_samples[:, component], bins=50, density=True, alpha=0.6, label="samples")
    grid = np.linspace(*ax.get_xlim(), 200)
    analytic_density = norm.pdf(
        grid, posterior_mean[component], posterior_standard_deviation[component]
    )
    ax.plot(grid, analytic_density, color="black", label="analytic")
    ax.set_xlabel(f"$u_{component}$")
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("Preconditioned pMALA: 1D marginals vs. analytic Gaussian")
fig.tight_layout()

## Pairwise 2D Marginals vs. Analytical (Preconditioned pMALA)

Reusing `add_covariance_ellipses` from above.

In [ ]:
fig, axes = plt.subplots(STATE_DIM, STATE_DIM, figsize=(3 * STATE_DIM, 3 * STATE_DIM))
for row in range(STATE_DIM):
    for col in range(STATE_DIM):
        ax = axes[row, col]
        if row <= col:
            ax.axis("off")
            continue
        ax.scatter(informed_samples[:, col], informed_samples[:, row], s=2, alpha=0.15)
        add_covariance_ellipses(
            ax,
            posterior_mean[[col, row]],
            posterior_covariance[np.ix_([col, row], [col, row])],
        )
        if row == STATE_DIM - 1:
            ax.set_xlabel(f"$u_{col}$")
        if col == 0:
            ax.set_ylabel(f"$u_{row}$")
fig.suptitle("Preconditioned pMALA: pairwise 2D marginals vs. analytic covariance ellipses")
fig.tight_layout()

## Step-Width Sweep

Sweeping $\delta$ over several orders of magnitude for both preconditioners makes the benefit
concrete: the informed preconditioner sustains a high acceptance rate across a much wider range of
step widths than the prior-preconditioned baseline.

In [ ]:
def sweep_acceptance_rate(
    preconditioner: GaussianMeasure, step_widths: np.ndarray
) -> np.ndarray:
    """Return the final running-mean acceptance rate for each step width in `step_widths`, from a
    short chain (fine for an acceptance-rate estimate, unlike the moment-recovery runs above)."""
    acceptance_rates = np.empty_like(step_widths)
    for index, step_width in enumerate(step_widths):
        _, acceptance_rates[index] = run_pmala_chain(preconditioner, float(step_width), seed=1)
    return acceptance_rates


sweep_step_widths = np.array([0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0])
plain_acceptance_sweep = sweep_acceptance_rate(reference_measure, sweep_step_widths)
informed_acceptance_sweep = sweep_acceptance_rate(informed_preconditioner, sweep_step_widths)

fig, ax = plt.subplots()
ax.plot(
    sweep_step_widths,
    plain_acceptance_sweep,
    marker="o",
    label=r"$\overline{K} = C$ (plain MALA)",
)
ax.plot(
    sweep_step_widths,
    informed_acceptance_sweep,
    marker="o",
    label=r"$\overline{K} = $ posterior covariance",
)
ax.set_xscale("log")
ax.set_xlabel(r"step width $\delta$")
ax.set_ylabel("acceptance rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title("pMALA acceptance rate vs. step width")
ax.legend()
fig.tight_layout()

## Summary

- `PMALAAlgorithm` keeps $\Phi$ relative to the target's true reference measure $\mu_0$
  throughout, using `model.reference` for $C^{-1}$, and `model.approximation` purely as a
  fixed preconditioner $\overline K$ for the Langevin proposal.
- With `approximation` equal to `reference` (i.e. $\overline K = C$, plain MALA), both runs above recovered the
  analytic posterior mean and standard deviation to within Monte Carlo error, but the acceptance
  rate collapses quickly as $\delta$ grows once the posterior's geometry departs from the prior's.
- Preconditioning with an informed $\overline K$ (here, the exact posterior covariance) sustains a
  high acceptance rate across step widths roughly two orders of magnitude larger, while still
  recovering the same posterior moments -- the practical payoff of Beskos et al.'s (2017)
  generalization.